# 3-03 거래대금 급증·변동성 — 갑자기 돈이 몰리거나 요동친 종목

**질문 예시:** "요즘 거래대금이 갑자기 터진 종목 있어? 평소 대비 몇 배인지랑 공시도 붙여줘",
"최근 하루 변동성이 제일 큰 종목 뽑아줘"

방향(등락)이 아니라 **관심의 폭발**(거래대금 급증)과 **불안정성**(일간 변동성)으로 종목을 고르고,
각 종목의 공시를 붙여 원인을 확인합니다.

**각색한 점:** CHECK 원본(`3_시세스크리너/03`)은 종목별 기간 시세를 돌며 계산했습니다.
DeepSearch는 `GetStockPrices`에 종목을 생략하면 **전 종목 하루치**가 한 번에 오므로,
거래일마다 한 번씩(25회) 불러 전 종목 패널을 만듭니다. 그래서 코스닥도 상위권이 아니라 **전수**입니다.
거래대금은 KRX 기준(NXT 미포함)입니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/3_price_screener/03_value_surge_volatility.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        # 한 페이지가 count 보다 적게 와도(80~90건 흔함) 뒤에 더 있을 수 있다 → 총건수로 판단
        if not docs or page * count >= total:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"   # 기준일
RECENT = 5            # 최근 구간(거래일)
BASE = 20             # 비교 기준 구간(최근 구간 직전 거래일)
MIN_RECENT_VALUE = 5e9   # 최근 구간 평균 거래대금 하한(50억) — 거래가 거의 없는 종목의 배수 착시 방지
TOP = 10

## 1. 거래일 목록과 전 종목 패널
거래일은 지수 시세(`GetMarketIndexes`)의 날짜로 정합니다. 거래일마다 전 종목 스냅샷을 불러 이어 붙입니다.

In [4]:
idx = index_prices((pd.Timestamp(ASOF) - pd.Timedelta(days=60)).date(), ASOF)
days = sorted(idx["date"].dt.date.unique())[-(RECENT + BASE):]
print(f"거래일 {len(days)}개: {days[0]} ~ {days[-1]}")

def stocks(day):
    """일반법인(1)·공기업(2) 보통주 전 종목 하루치. 헬퍼의 all_stocks(common_only=True)는 공기업 8종목
    (한국전력·기업은행·강원랜드 등, company_type_l1='2')을 빼므로 여기서 직접 거릅니다."""
    s = all_stocks(day, common_only=False)
    return s[s["company_type_l1"].isin(["1", "2"]) & s["symbol"].str.endswith("0")].reset_index(drop=True)

frames = []
for d in days:
    s = stocks(d)
    frames.append(s[s["date"].dt.date == d])   # 휴장일 대체 방지: 그날 행만
panel = pd.concat(frames, ignore_index=True)
panel = panel[panel["volume"] > 0]              # 거래정지일 제외
print(panel.shape, "종목 수", panel["symbol"].nunique())

거래일 25개: 2026-10-21 ~ 2026-11-24


(65505, 20) 종목 수 2666


## 2. 지표 계산
- **거래대금 급증 배수** = 최근 5거래일 평균 거래대금 ÷ 직전 20거래일 평균 거래대금
- **최대 일간 변동** = 최근 5거래일 중 하루 |등락률| 최댓값
- **평균 일중 폭** = (고가 − 저가) ÷ 전일 종가의 최근 5일 평균

25거래일 중 거래일이 20일 미만인 종목(신규상장·장기 거래정지)은 뺍니다.

In [5]:
recent_days, base_days = set(days[-RECENT:]), set(days[:BASE])
panel["d"] = panel["date"].dt.date
panel["prev_close"] = panel["close"] / (1 + panel["change_rate"])
panel["range_pct"] = (panel["high"] - panel["low"]) / panel["prev_close"] * 100

g = panel.groupby("symbol")
m = pd.DataFrame({
    "name": g["name"].last(), "market": g["market"].last(),
    "n_days": g["d"].nunique(),
    "recent_value": panel[panel["d"].isin(recent_days)].groupby("symbol")["value"].mean(),
    "base_value": panel[panel["d"].isin(base_days)].groupby("symbol")["value"].mean(),
    "max_abs_move": panel[panel["d"].isin(recent_days)].groupby("symbol")["change_pct"].apply(lambda s: s.abs().max()),
    "limit_days": panel[panel["d"].isin(recent_days)].groupby("symbol")["change_pct"].apply(lambda s: int((s.abs() >= 29.5).sum())),
    "avg_range": panel[panel["d"].isin(recent_days)].groupby("symbol")["range_pct"].mean(),
    "ret_recent": panel[panel["d"].isin(recent_days)].groupby("symbol")["change_rate"].apply(lambda s: ((1 + s).prod() - 1) * 100),
})
m = m[(m["n_days"] >= 20) & m["base_value"].gt(0)].copy()
m["limit_days"] = m["limit_days"].astype(int)
m["surge_x"] = m["recent_value"] / m["base_value"]
print("계산 대상 종목:", len(m))

계산 대상 종목: 2592


In [6]:
m["recent_eok"] = m["recent_value"] / 1e8   # 억원
m["base_eok"] = m["base_value"] / 1e8
surge = m[m["recent_value"] >= MIN_RECENT_VALUE].sort_values("surge_x", ascending=False).head(TOP).copy()
print(PRICE_NOTICE)
print("거래대금 급증 상위 (금액 단위: 억원, 수익률·변동: %)")
surge[["name", "market", "surge_x", "recent_eok", "base_eok", "ret_recent", "max_abs_move"]].round(1)

※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). 거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).
거래대금 급증 상위 (금액 단위: 억원, 수익률·변동: %)


,name,market,surge_x,recent_eok,base_eok,ret_recent,max_abs_move
symbol,,,,,,,
KRX:451250,삐아,KOSDAQ,260.0,132.1,0.5,46.5,30.0
KRX:123690,한국화장품,KOSPI,89.2,173.1,1.9,39.5,30.0
KRX:439090,마녀공장,KOSDAQ,15.9,188.6,11.9,21.2,18.7
KRX:253450,스튜디오드래곤,KOSDAQ,13.7,221.0,16.2,47.9,30.0
KRX:090710,휴림로봇,KOSDAQ,12.4,434.6,35.1,71.2,30.0
KRX:446840,지슨,KOSDAQ,12.1,198.6,16.5,-9.8,29.9
KRX:039130,하나투어,KOSPI,9.5,112.5,11.8,5.7,16.8
KRX:004990,롯데지주,KOSPI,8.6,178.2,20.6,-0.4,4.5
KRX:037270,YG PLUS,KOSPI,8.0,127.7,16.0,7.3,12.7


상·하한가(±30%)가 많아 최대 일간 변동만으로는 동점이 생깁니다. **상·하한가 일수 → 최대 일간 변동 → 평균 일중 폭** 순으로 정렬합니다.

In [7]:
vol = (m[m["recent_value"] >= 1e9]
       .sort_values(["limit_days", "max_abs_move", "avg_range"], ascending=False).head(TOP).copy())
print("변동성 상위 (최근 5거래일, %)")
vol[["name", "market", "limit_days", "max_abs_move", "avg_range", "ret_recent", "surge_x"]].round(1)

변동성 상위 (최근 5거래일, %)


,name,market,limit_days,max_abs_move,avg_range,ret_recent,surge_x
symbol,,,,,,,
KRX:355390,크라우드웍스,KOSDAQ,2,30.0,16.7,-57.2,8.7
KRX:090710,휴림로봇,KOSDAQ,2,30.0,14.3,71.2,12.4
KRX:199730,바이오인프라,KOSDAQ,2,30.0,24.2,35.5,1.5
KRX:208640,썸에이지,KOSDAQ,2,29.9,21.1,81.7,4.7
KRX:129920,대성하이텍,KOSDAQ,1,30.0,15.3,36.5,1.8
KRX:282720,금양그린파워,KOSDAQ,1,30.0,11.1,35.8,1.9
KRX:241840,에이스토리,KOSDAQ,1,30.0,7.8,-7.1,17.5
KRX:207760,미스터블루,KOSDAQ,1,30.0,15.8,29.9,116.0
KRX:261780,아리바이오랩,KOSDAQ,1,30.0,18.3,-6.4,7.3


## 3. 공시 붙이기
두 목록의 종목을 묶어 **한 번의 공시 검색**(`securities.symbol:(코드 or 코드 …)`)으로 최근 구간 공시를 받습니다.
회사가 낸 공시와 거래소가 붙인 경보(투자주의·투자경고·공매도 과열 등)가 함께 나옵니다.

In [8]:
def attach_disclosures(symbols, date_from, date_to, per_stock=4):
    codes = [s.split(":")[-1] for s in symbols]
    total, docs = doc_search("company", ["disclosure"], "securities.symbol:(" + " or ".join(codes) + ")",
                             date_from, date_to, count=100, pages=3)
    out = {}
    if docs.empty:   # 0건이면 doc_search 가 열 없는 표를 돌려줌
        return {"KRX:" + c: (0, "(공시 없음)") for c in codes}
    for code in codes:
        sub = docs[docs["symbols"].str.contains(code, na=False)].sort_values("created_at")
        titles = sub["title"].str.replace(r"\s+", " ", regex=True).drop_duplicates().tolist()
        out["KRX:" + code] = (len(sub), " → ".join(titles[-per_stock:]) if titles else "(공시 없음)")
    return out

syms = list(dict.fromkeys(list(surge.index) + list(vol.index)))
disc = attach_disclosures(syms, days[-RECENT], ASOF)
for name, tbl in (("거래대금 급증", surge), ("변동성", vol)):
    tbl["공시수"] = [disc[s][0] for s in tbl.index]
    tbl["최근 공시(오래된→최근)"] = [disc[s][1] for s in tbl.index]

surge[["name", "surge_x", "공시수", "최근 공시(오래된→최근)"]].round(1)

,name,surge_x,공시수,최근 공시(오래된→최근)
symbol,,,,
KRX:451250,삐아,260.0,0,(공시 없음)
KRX:123690,한국화장품,89.2,0,(공시 없음)
KRX:439090,마녀공장,15.9,0,(공시 없음)
KRX:253450,스튜디오드래곤,13.7,3,공매도 과열종목 지정(공매도 거래 금지 적용) → (예고)단기과열종목 지정예고
KRX:090710,휴림로봇,12.4,2,[투자주의]투자경고종목 지정예고 → 공매도 과열종목 지정(공매도 거래 금지 적용)
KRX:446840,지슨,12.1,1,투자경고종목지정
KRX:039130,하나투어,9.5,0,(공시 없음)
KRX:004990,롯데지주,8.6,4,최대주주등소유주식변동신고서 → 대표이사(대표집행임원)변경(안내공시) → 타인에대한채무보증결정(자회사의...
KRX:037270,YG PLUS,8.0,0,(공시 없음)


In [9]:
vol[["name", "max_abs_move", "공시수", "최근 공시(오래된→최근)"]].round(1)

,name,max_abs_move,공시수,최근 공시(오래된→최근)
symbol,,,,
KRX:355390,크라우드웍스,30.0,8,조회공시요구(풍문또는보도)에대한답변(부인) → 불성실공시법인지정예고(공시불이행) → 불성실공시법인지정...
KRX:090710,휴림로봇,30.0,2,[투자주의]투자경고종목 지정예고 → 공매도 과열종목 지정(공매도 거래 금지 적용)
KRX:199730,바이오인프라,30.0,0,(공시 없음)
KRX:208640,썸에이지,29.9,12,[투자주의]투자경고종목 지정예고 → 기타시장안내 (주가 미달에 따른 상장폐지 우려 관련 안내) → 기...
KRX:129920,대성하이텍,30.0,2,주식등의대량보유상황보고서(일반) → [투자주의]상한가잔량 상위종목
KRX:282720,금양그린파워,30.0,1,단일판매ㆍ공급계약체결
KRX:241840,에이스토리,30.0,2,공매도 과열종목 지정(공매도 거래 금지 적용) → 주식등의대량보유상황보고서(일반)
KRX:207760,미스터블루,30.0,0,(공시 없음)
KRX:261780,아리바이오랩,30.0,3,최대주주변경을수반하는주식담보제공계약해제ㆍ취소등 → 최대주주변경을수반하는주식담보제공계약체결 → [기재정...


## 결과 읽는 법과 한계
- **배수는 "최근 거래가 평소의 몇 배인가"**입니다. 기준 구간 거래가 거의 없던 종목은 배수가 과장되므로
  최근 평균 거래대금 하한(`MIN_RECENT_VALUE`)을 둡니다.
- **최대 일간 변동이 30%에 가까우면 상·하한가**입니다. 거래정지 후 재개 첫날은 30%를 넘을 수 있습니다.
- 공시가 없으면 본주 공시 없이 테마·뉴스로 움직인 경우입니다 → `08_movers_with_reasons`에서 뉴스까지 붙여 보세요.
- 가격은 수정주가라 기간 중 무상증자·분할이 있어도 등락률이 왜곡되지 않습니다. 대신 표의 가격은 당시 호가와 다를 수 있습니다.
- 거래대금은 KRX만 집계되어 NXT 거래분만큼 작게 나옵니다(대형주는 절반 가까이).